# Data Cleaning 04 -- IBES Price Targets

## Input
`Data/Data_Collection/Initial/04_LSEG_IBES/ibes_price_targets.parquet` (1,380,407 rows across 14,971 IBES tickers)

## Purpose
Cleans monthly stock-level analyst price target data from WRDS IBES. The raw file contains the entire IBES US coverage universe. This notebook filters to the top-100 S&P 500 universe, then validates value ranges, NaN patterns, and structural consistency.

## Stage 0: Load, Filter to Universe, Inspect
- Raw file filtered from 14,971 tickers to the 231 tickers mapping to the 227 master PERMNOs using `ibes_permno_link_clean.parquet`
- Trimmed to 2004-01-01 onwards
- Reduced from 1,380,407 to 49,310 rows
- Basic shape, date range, ticker counts, and per-ticker coverage statistics reported

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts with flags for columns above 30%
- Per-row NaN distribution

## Stage 2: Value Range & Quality Checks
- **Price target levels:** range checks on `ptg_mean`, `ptg_median`, `ptg_high`, `ptg_low` in dollar terms, flags values above $2,000
- **Analyst count:** distribution of `ptg_numest`, fraction of single-analyst observations
- **Revision columns:** distribution and percentile analysis of `ptg_revision` and `ptg_revision_3m`, counts extreme values (>100% or <-50%)
- **Dispersion and range:** percentile analysis of `ptg_dispersion` and `ptg_range`
- **Upside skew:** range, infinite values, and extreme value counts for `ptg_upside_skew`
- **Consistency checks:** verifies `ptg_high >= ptg_low` and `ptg_mean` falls within `[ptg_low, ptg_high]`
- **Coverage per ticker:** months of data per ticker, identifies tickers with fewer than 24 months
- **Duplicate (ticker, date) check**
- **NaN pattern analysis:** confirms `ptg_dispersion` NaN occurs exclusively when `ptg_numest = 1`, and `ptg_upside_skew` NaN occurs exclusively when `ptg_median = ptg_low` (zero denominator)

## Stage 4: Clean & Save

### Filtered to Universe (Stage 0)
Reduced from 1,380,407 rows (14,971 tickers) to 49,310 rows (231 tickers mapping to 227 PERMNOs), trimmed to 2004+.

### No Columns Dropped
All 11 factor columns retained. Raw dollar columns (`ptg_mean`, `ptg_median`, `ptg_high`, `ptg_low`) are kept because all factors will be z-standardised cross-sectionally per date in the merge pipeline, making absolute scale irrelevant. `ptg_mean` is also needed for computing implied return (`ptg_mean / current_price - 1`) during the merge.

### No Winsorisation Applied Here
Extreme values in `ptg_revision` (up to +300%), `ptg_revision_3m` (up to +450%), and `ptg_upside_skew` (up to 142.8) are left as-is. Winsorisation will be applied once in the merge pipeline, cross-sectionally per date, uniformly across all factors, right before z-standardisation. This keeps the logic in one place and makes the threshold easy to change.

### Structural NaN Left as NaN (745 Total, <0.14% of Cells)
Every NaN has a specific structural cause -- none are data quality issues:
- `ptg_dispersion` (269 NaN): 100% occur where `ptg_numest = 1`. Dispersion requires 2+ analysts; with one analyst it is undefined.
- `ptg_upside_skew` (326 NaN): 100% occur where `ptg_median = ptg_low`. The formula `(high - median) / (median - low)` has a zero denominator.
- `ptg_revision` (30 NaN), `ptg_numest_chg` (30 NaN): first observation per ticker, no prior month to compute a change from.
- `ptg_revision_3m` (90 NaN): first 3 observations per ticker.

These will be skipped during cross-sectional aggregation in the merge pipeline.

## Output
`Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_price_targets_clean.parquet` -- 11 factor columns (all retained), 49,310 rows

In [2]:
# %% [markdown]
# # Data Cleaning: ibes_price_targets.parquet
#
# Source: Data/Data_Collection/Initial/04_LSEG_IBES/ibes_price_targets.parquet
# Output: Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_price_targets_clean.parquet
#
# Monthly stock-level analyst price target data from WRDS IBES.
# The raw file contains ALL ~15,000 IBES tickers. We filter to our
# 227-PERMNO universe first using the clean link table, then run diagnostics
# on the filtered data only.

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH  = Path('../../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_price_targets.parquet')
LINK_PATH = Path('../../../Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_permno_link_clean.parquet')
OUT_DIR   = Path('../../../Data/Data_Collection/Cleaned/04_LSEG_IBES')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
link = pd.read_parquet(LINK_PATH)

print(f"\n  Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Raw tickers: {df['ticker'].nunique():,}")

# ── Filter to universe tickers ───────────────────────────────────────────────
valid_tickers = set(link['ticker'].unique())
n_before = len(df)
df = df[df['ticker'].isin(valid_tickers)].reset_index(drop=True)
print(f"\n  Filtered to universe tickers: {n_before:,} → {len(df):,} rows")
print(f"  Tickers retained: {df['ticker'].nunique()}")

# ── Trim to 2004-01-01 (match rest of pipeline) ─────────────────────────────
n_before = len(df)
df = df[df['date'] >= '2004-01-01'].reset_index(drop=True)
print(f"  Trimmed to 2004+: {n_before:,} → {len(df):,} rows")

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique()}")
print(f"  Unique tickers: {df['ticker'].nunique()}")
print(f"  Rows per ticker (mean): {df.groupby('ticker').size().mean():.1f}")
print(f"  Rows per ticker (median): {df.groupby('ticker').size().median():.0f}")

factor_cols = [c for c in df.columns if c not in ['ticker', 'date']]

print(f"\nColumns and dtypes ({len(factor_cols)} factors):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<25s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
print(df.head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df.tail(10).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN ───────────────────────────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<25s} {'NaN %':>8s}  {'Count':>8s}")
print("  " + "-" * 45)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    flag = " ← DROP" if pct >= 30 else ""
    print(f"  {col:<25s} {pct:>7.2f}%  {count:>8,d}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>8,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-3 NaN: {((row_nan >= 1) & (row_nan <= 3)).sum():>8,d}")
print(f"  Rows with 4-6 NaN: {((row_nan > 3) & (row_nan <= 6)).sum():>8,d}")
print(f"  Rows with >6 NaN: {(row_nan > 6).sum():>8,d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: VALUE RANGE & QUALITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: VALUE RANGE & QUALITY CHECKS")
print("=" * 90)

# ── 2a. Price target levels ──────────────────────────────────────────────────
print(f"\n--- Price target levels (dollar values) ---")
for col in ['ptg_mean', 'ptg_median', 'ptg_high', 'ptg_low']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"  {col:<20s} min: ${vals.min():.2f}  median: ${vals.median():.2f}  "
          f"max: ${vals.max():.2f}  mean: ${vals.mean():.2f}")
    n_extreme = (vals > 2000).sum()
    if n_extreme > 0:
        print(f"    ⚠ {n_extreme} values above $2,000")

# ── 2b. Analyst count ────────────────────────────────────────────────────────
if 'ptg_numest' in df.columns:
    vals = df['ptg_numest'].dropna()
    print(f"\n--- Analyst count (ptg_numest) ---")
    print(f"  range: {vals.min():.0f} – {vals.max():.0f}  "
          f"mean: {vals.mean():.1f}  median: {vals.median():.0f}")
    print(f"  Stocks with only 1 analyst: {(vals == 1).sum():,} "
          f"({(vals == 1).mean()*100:.1f}%)")

# ── 2c. Revision columns ────────────────────────────────────────────────────
print(f"\n--- Revision columns (% change) ---")
for col in ['ptg_revision', 'ptg_revision_3m']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"  {col}:")
    print(f"    range: {vals.min():.2f}% – {vals.max():.2f}%")
    print(f"    mean: {vals.mean():.2f}%, median: {vals.median():.2f}%")
    pctiles = vals.quantile([0.01, 0.05, 0.95, 0.99])
    print(f"    1st: {pctiles[0.01]:.2f}%  5th: {pctiles[0.05]:.2f}%  "
          f"95th: {pctiles[0.95]:.2f}%  99th: {pctiles[0.99]:.2f}%")
    n_extreme = ((vals > 100) | (vals < -50)).sum()
    print(f"    Extreme (>100% or <-50%): {n_extreme}")

# ── 2d. Dispersion and range ────────────────────────────────────────────────
print(f"\n--- Dispersion and range ---")
for col in ['ptg_dispersion', 'ptg_range']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    pctiles = vals.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
    print(f"  {col}:")
    print(f"    range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"    1st: {pctiles[0.01]:.4f}  25th: {pctiles[0.25]:.4f}  "
          f"median: {pctiles[0.50]:.4f}  75th: {pctiles[0.75]:.4f}  "
          f"99th: {pctiles[0.99]:.4f}")

# ── 2e. Upside skew ─────────────────────────────────────────────────────────
if 'ptg_upside_skew' in df.columns:
    vals = df['ptg_upside_skew'].dropna()
    print(f"\n--- ptg_upside_skew ---")
    print(f"  range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"  mean: {vals.mean():.4f}, median: {vals.median():.4f}")
    n_inf = np.isinf(vals).sum()
    n_extreme = (vals.abs() > 10).sum()
    print(f"  Infinite values: {n_inf}")
    print(f"  |skew| > 10: {n_extreme}")
    pctiles = vals.quantile([0.01, 0.05, 0.95, 0.99])
    print(f"  1st: {pctiles[0.01]:.4f}  5th: {pctiles[0.05]:.4f}  "
          f"95th: {pctiles[0.95]:.4f}  99th: {pctiles[0.99]:.4f}")

# ── 2f. Consistency checks ──────────────────────────────────────────────────
print(f"\n--- Consistency checks ---")
if all(c in df.columns for c in ['ptg_high', 'ptg_low', 'ptg_mean', 'ptg_median']):
    both_valid = df[['ptg_high', 'ptg_low', 'ptg_mean', 'ptg_median']].dropna()
    n_inverted = (both_valid['ptg_high'] < both_valid['ptg_low']).sum()
    n_mean_outside = (
        (both_valid['ptg_mean'] < both_valid['ptg_low']) |
        (both_valid['ptg_mean'] > both_valid['ptg_high'])
    ).sum()
    print(f"  ptg_high < ptg_low (inverted): {n_inverted}")
    print(f"  ptg_mean outside [ptg_low, ptg_high]: {n_mean_outside}")

# ── 2g. Coverage per ticker ─────────────────────────────────────────────────
print(f"\n--- Coverage per ticker ---")
ticker_coverage = df.groupby('ticker').agg(
    n_months=('date', 'nunique'),
    first_date=('date', 'min'),
    last_date=('date', 'max')
)
print(f"  Months per ticker: mean={ticker_coverage['n_months'].mean():.0f}, "
      f"median={ticker_coverage['n_months'].median():.0f}, "
      f"min={ticker_coverage['n_months'].min()}, "
      f"max={ticker_coverage['n_months'].max()}")

short_coverage = ticker_coverage[ticker_coverage['n_months'] < 24]
if len(short_coverage) > 0:
    print(f"\n  Tickers with <24 months: {len(short_coverage)}")
    for ticker, row in short_coverage.iterrows():
        print(f"    {ticker:<10s} {row['n_months']:>3d} months  "
              f"({row['first_date'].date()} → {row['last_date'].date()})")

# ── 2h. Duplicate (ticker, date) ────────────────────────────────────────────
print(f"\n--- Duplicate (ticker, date) ---")
n_dupes = df.duplicated(subset=['ticker', 'date']).sum()
if n_dupes == 0:
    print(f"  ✓ No duplicates")
else:
    print(f"  ⚠ {n_dupes} duplicates")

# ── 2i. NaN pattern analysis ────────────────────────────────────────────────
print(f"\n--- NaN pattern: ptg_dispersion when numest = 1 ---")
if all(c in df.columns for c in ['ptg_dispersion', 'ptg_numest']):
    single_analyst = df['ptg_numest'] == 1
    disp_nan = df['ptg_dispersion'].isna()
    both = (single_analyst & disp_nan).sum()
    disp_nan_total = disp_nan.sum()
    print(f"  ptg_dispersion NaN total: {disp_nan_total}")
    print(f"  Of those, where ptg_numest = 1: {both} ({both/max(disp_nan_total,1)*100:.1f}%)")

print(f"\n--- NaN pattern: ptg_upside_skew when median = low ---")
if all(c in df.columns for c in ['ptg_upside_skew', 'ptg_median', 'ptg_low']):
    median_eq_low = df['ptg_median'] == df['ptg_low']
    skew_nan = df['ptg_upside_skew'].isna()
    both = (median_eq_low & skew_nan).sum()
    skew_nan_total = skew_nan.sum()
    print(f"  ptg_upside_skew NaN total: {skew_nan_total}")
    print(f"  Of those, where median = low: {both} ({both/max(skew_nan_total,1)*100:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: SUMMARY — DECISIONS NEEDED")
print("=" * 90)

print(f"""
Now that the data is filtered to your universe, review:

1. COLUMNS TO DROP:
   - ptg_upside_skew if still has extreme values (unstable ratio)
   - Any column with ≥30% NaN

2. COLUMNS TO KEEP vs DROP (dollar values):
   - ptg_mean: KEEP (needed for implied return = ptg_mean/price - 1)
   - ptg_median, ptg_high, ptg_low: useful for ptg_dispersion and ptg_range
     (which are already computed). Consider dropping the raw dollar values
     and keeping only the normalised ratios.
   - ptg_numest: KEEP (analyst coverage signal)
   - ptg_numest_chg: KEEP (change in coverage)
   - ptg_revision, ptg_revision_3m: KEEP but may need winsorising

3. EXTREME VALUES:
   - ptg_revision/ptg_revision_3m: winsorise at 1st/99th percentile?
   - ptg_upside_skew: clip at reasonable bounds or drop?

4. NaN HANDLING:
   - ptg_dispersion NaN when ptg_numest = 1 (only one analyst, no dispersion)
   - ptg_upside_skew NaN when ptg_median = ptg_low (division by zero)
   - ptg_revision NaN in first month per ticker (no prior month)
   - ptg_revision_3m NaN in first 3 months per ticker

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT

  Raw shape: 1,380,407 rows × 13 columns
  Raw tickers: 14,971

  Filtered to universe tickers: 1,380,407 → 51,660 rows
  Tickers retained: 230
  Trimmed to 2004+: 51,660 → 49,310 rows

  Shape: 49,310 rows × 13 columns
  Date range: 2004-01-31 → 2024-12-31
  Unique dates: 252
  Unique tickers: 230
  Rows per ticker (mean): 214.4
  Rows per ticker (median): 252

Columns and dtypes (11 factors):
    1. ptg_mean                  Float64        
    2. ptg_median                Float64        
    3. ptg_high                  Float64        
    4. ptg_low                   Float64        
    5. ptg_numest                Int64          
    6. ptg_dispersion            Float64        
    7. ptg_range                 Float64        
    8. ptg_upside_skew           Float64        
    9. ptg_revision              Float64        
   10. ptg_revision_3m           Float64        
   11. ptg_numest_chg            Int64          

--- Head (10 rows)

In [3]:
# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Filtered to universe (Stage 0):**
# The raw file contains 1,380,407 rows across 14,971 IBES tickers — the entire
# IBES US coverage universe. Filtered to the 231 tickers that map to our 227
# top-100 S&P 500 PERMNOs via the clean link table, then trimmed to 2004+.
# This reduced the data from 1.38M rows to 49,310 rows.
#
# **No columns dropped.** All 11 factor columns retained. Raw dollar columns
# will be z-standardised cross-sectionally in the merge pipeline.
#
# **No winsorisation applied here.** Extreme values in `ptg_revision` (up to
# +300%), `ptg_revision_3m` (up to +450%), and `ptg_upside_skew` (up to 142.8)
# are left as-is. Winsorisation will be applied once, uniformly across all
# factors, cross-sectionally per date, in the merge pipeline — right before
# z-standardisation. This avoids scattering winsorisation logic across multiple
# notebooks and allows the threshold to be changed in one place.
#
# **Structural NaN left as NaN (745 total, <0.14% of cells):**
# Every NaN has a specific structural cause — none are data quality issues:
# - `ptg_dispersion` (269 NaN): 100% occur where `ptg_numest = 1`.
#   Dispersion requires ≥2 analysts; with one analyst, it is undefined.
# - `ptg_upside_skew` (326 NaN): 100% occur where `ptg_median = ptg_low`.
#   The formula (high − median) / (median − low) has a zero denominator.
# - `ptg_revision` (30 NaN), `ptg_numest_chg` (30 NaN): first observation
#   per ticker — no prior month to compute a change from.
# - `ptg_revision_3m` (90 NaN): first 3 observations per ticker.
# These will be skipped during cross-sectional aggregation.
#
# **Factors retained: 11** (all kept)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

factor_cols = [c for c in df.columns if c not in ['ticker', 'date']]

# ── 4a. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols].isna().sum()
nan_cols = nan_check[nan_check > 0]
if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN")
else:
    total_nan = nan_cols.sum()
    print(f"\n  Remaining NaN: {total_nan} (structural, left intentionally)")
    for col, n in nan_cols.items():
        print(f"    {col:<25s} {n:>5d} NaN")

# ── 4b. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Tickers: {df['ticker'].nunique()}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    vals = df[c].dropna()
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n} NaN)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<25s} range: [{vals.min():.4f}, {vals.max():.4f}]{nan_str}")

print(f"\n  Sample (first 5 rows):")
print(df.head(5).to_string(index=False))

# ── 4c. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'ibes_price_targets_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 4: CLEAN & SAVE

  Remaining NaN: 745 (structural, left intentionally)
    ptg_dispersion              269 NaN
    ptg_upside_skew             326 NaN
    ptg_revision                 30 NaN
    ptg_revision_3m              90 NaN
    ptg_numest_chg               30 NaN

  Final shape: 49,310 rows × 13 columns
  Tickers: 230
  Date range: 2004-01-31 → 2024-12-31

  Factor list (11 columns):
      1. ptg_mean                  range: [0.1314, 5100.6380]
      2. ptg_median                range: [0.1180, 5100.0000]
      3. ptg_high                  range: [0.2580, 6120.0000]
      4. ptg_low                   range: [0.0100, 3900.0000]
      5. ptg_numest                range: [1.0000, 71.0000]
      6. ptg_dispersion            range: [0.0000, 1.6994]  (269 NaN)
      7. ptg_range                 range: [0.0000, 5.6325]
      8. ptg_upside_skew           range: [0.0000, 142.8000]  (326 NaN)
      9. ptg_revision              range: [-80.8307, 300.0000]  (30 NaN)
     10. ptg_revis